# Exploring sequence transforms

Compare transforms of the same residue channels. These are illustrative sequences,
not a supervised benchmark. Edit the controls, then rerun downstream cells.
Fourier/Z axes and Mellin log-frequency axes have different units; wavelets use
scale bands and position indices. See [transform conventions](../docs/TRANSFORMS.md).

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "protspect").is_dir())
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from protspect import transform_sequence
from protspect.embeddings import kidera_factors
from protspect.sklearn import SequenceTransformer
%matplotlib inline

In [ ]:
# Replace this mapping with any AAindex or custom representation.
EMBEDDING = kidera_factors()
SEQUENCES = ["GIGKFLHSAKKFGKAFVGEIMNS", "GIGKFLHSAKKFGKAFVGEIMNS"[::-1]]
CHANNEL = "KF1"
LENGTH = 64
CENTER = True
NORMALIZATION = "energy"
Z_RADIUS = 1.05
MELLIN_C = 0.0
WAVELET = "db2"
LEVEL = 3
MODE = "symmetric"
COMPONENT = "real"  # amplitude, phase, real, imaginary
FIGSIZE = (12, 9)

In [ ]:
OPTIONS = {"fourier": {}, "z": {"radius": Z_RADIUS},
           "mellin": {"mellin_c": MELLIN_C},
           "wavelet": {"wavelet": WAVELET, "level": LEVEL, "mode": MODE}}
results = {method: [transform_sequence(s, EMBEDDING, method=method,
                length=LENGTH, center=CENTER, normalization=NORMALIZATION, **options)
                for s in SEQUENCES] for method, options in OPTIONS.items()}
display(pd.DataFrame([r[0].metadata for r in results.values()]).fillna(""))

In [ ]:
component = {"amplitude": np.abs, "phase": np.angle,
             "real": np.real, "imaginary": np.imag}[COMPONENT]
fig, axes = plt.subplots(2, 2, figsize=FIGSIZE, constrained_layout=True)
for ax, (method, outputs) in zip(axes.flat, results.items()):
    for i, result in enumerate(outputs):
        values = component(result.coefficients[CHANNEL])
        key = "log_frequency" if method == "mellin" else "frequency"
        x = result.coordinates[key] if method != "wavelet" else np.arange(len(values))
        ax.plot(x, values, label=f"Sequence {i+1}", alpha=.8)
    ax.set_title(f"{method}: {CHANNEL}")
    ax.set_xlabel({"mellin": "Log-frequency ν", "wavelet": "Concatenated band coefficient"}.get(method, "Cycles per residue"))
    ax.set_ylabel(COMPONENT)
    ax.legend()
plt.show()
wave = results["wavelet"][0]
display(pd.DataFrame(wave.coordinates).groupby("band", sort=False).agg(coefficient_count=("index", "size")))

## Fixed-width pipeline output

Choose a method and rerun this cell. Actual modelling should fit inside each CV
fold. Amplitude is not a signed wavelet coefficient. Radius 1 should match Fourier;
radii away from 1 change positional weights. Mellin here is a finite weighted sum
on positive positions, not a scale-invariance guarantee.

In [ ]:
METHOD = "wavelet"
estimator = SequenceTransformer(embedding=EMBEDDING, method=METHOD,
    length=LENGTH, center=CENTER, normalization=NORMALIZATION,
    component=COMPONENT, transform_options=OPTIONS[METHOD])
X = estimator.fit_transform(SEQUENCES)
print("Shape:", X.shape)
display(pd.DataFrame(X, columns=estimator.get_feature_names_out()).iloc[:, :12])